# Filtres avancés et Reranking

## Filtres
Qdrant peut également filtrer les données sur base des champs présents de la *payload* comme nous pourrions le faire avec une clause ``WHERE`` en sql.

Ces filtres permettent de par exemple récupérer les chunks des documents les plus récentes ou bien de récupérer que les chunks qu'un utilisateur a le droit de voir.

C'est en combinant les recherches sémantiques, lexicales et ces filtres que nous récupèront un contexte de qualité pour notre agent.

Un RAG doit toujours chercher à optimiser deux métriques qui tirent dans des sens opposés.
- Le recall: il faut récupérer tous les chunks pertinents rapidement. Ce qui implique d'en récupérer certains qui ne le sont pas
- La precision: faire une sorte que les premiers chunks soient les bons

Les recherches sémantiques, lexicales, RRF et les filtres permettent en partie de résoudre cette équation mais ne suffisent pas toujours pour garantir que les chunks soient classés dans un bon ordre.

In [35]:
import os
from dotenv import load_dotenv

import numpy as np
import polars as pl

from docling_core.types.doc import DoclingDocument

import requests as rq
from openai import OpenAI
from qdrant_client import QdrantClient, models

from pathlib import Path

load_dotenv()

True

In [3]:
QDRANT_URL = 'http://127.0.0.1:6333'
QDRANT_API_KEY = os.getenv('QDRANT_API_KEY')
COLLECTION = 'Veltis'

qclient = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY
)

C:\Users\romainwauman\AppData\Local\Temp\ipykernel_110296\2070879123.py:5: UserWarning: Api key is used with an insecure connection.
  qclient = QdrantClient(


## La payload

le *payload_schema* ne donne que les champs qui sont indexés.

Afin de parcourir toutes les clefs réellement présentes, il faut parcourir tous les points à l'aide de ``scroll``.

In [4]:
qclient.get_collection(COLLECTION).payload_schema.keys()

dict_keys(['document'])

In [5]:
n_points = qclient.count(COLLECTION).count
n_points

85

In [6]:
offset = 50
keys = set()
for i in range(0, offset, n_points):
    points, offset = qclient.scroll(
        COLLECTION,
        limit = offset,
        offset=i,
        with_payload=True,
        with_vectors=False
    )
    for p in points:
        keys.update(p.payload.keys())
keys

{'chunk_index', 'document', 'file_extension', 'headings', 'text'}

Ensuite qdrant propose 3 conditions:
- ``must``: toutes les conditions doivent êtres respectées **AND**
- ``should``: au moins des conditions doit être respectée **OR**
- ``must_not``: aucune des conditions ne doit êtres respectées **NOT**

Qdrant propose toutes une serie de fonction pour la recherche:
- ``MatchValue``: pour une correspondance exact
- ``MatchText``: pour l'équivalent d'un *contains*
- ``Range``: pour l'équivalent d'un *between*

C'est également utile avec la méthode ``delete`` si vous devez réindexer un document:


```python
document_filter = models.Filter(must=[models.FieldCondition(key="document", match=models.MatchValue(value=name))])
qdrant.delete(COLLECTION, points_selector=models.FilterSelector(filter=document_filter))
index_chunks(chunk_document(name))
```

In [7]:
question = 'Quelles sont les données financières clefs de veltis ?'
filters = models.Filter(
    must = [
        models.FieldCondition(key = 'text', match=models.MatchText(text='plan'))
    ]
)

results = qclient.query_points(
    COLLECTION, using='sparse',
    limit = 10, query_filter=filters
)

pl.from_dicts(results.points)

id,version,score,payload,vector,shard_key,order_value
str,i64,f64,struct[5],null,null,null
"""456fb1a3-02e8-515e-a12c-ab3a3b…",0,1.0,"{""plan_strategique_2026-2028.pptx"",0,"""",""Plan stratégique « Cap 2028 » Comité de direction — janvier 2026 Où en sommes-nous ? 38,4 M€ 9 850 27 % chiffre d'affaires 2025 vélos vendus en 2025 du chiffre d'affaires à l'export Veltis SA — Comité de direction — confidentiel Trois axes stratégiques 1. Professionnels 2. Export 3. Services Devenir la référence du vélo de livraison urbaine auprès des entreprises. Ouvrir un nouveau pays par an en Europe du Nord. Entretien sur site, location longue durée et reconditionnement. Veltis SA — Comité de direction — confidentiel Trajectoire de croissance Chiffre d'affaires annuel, en millions d'euros Veltis SA — Comité de direction — confidentiel Calendrier d'ouverture des marchés export Pays-Bas, Année d'ouverture = 2026. Pays-Bas, Mode d'entrée = Distributeur exclusif. Allemagne, Année d'ouverture = 2027. Allemagne, Mode d'entrée = Filiale commerciale. Danemark, Année d'ouverture = 2028. Danemark, Mode d'entrée = Distributeur exclusif Veltis SA — Comité de direction — confidentiel Investissements clés • Deuxième site de production à Lorvan : 6,5 millions d'euros (sous réserve d'approbation du conseil d'administration) • Atelier de reconditionnement à Marnay : 0,8 million d'euros • Plateforme de suivi connecté des flottes professionnelles : 0,5 million d'euros Veltis SA — Comité de direction — confidentiel Prochaines étapes • Mars 2026 : présentation du projet de deuxième site au conseil d'administration • Avril 2026 : signature du contrat de distribution aux Pays-Bas • Juin 2026 : lancement de l'offre de location longue durée Veltis SA — Comité de direction — confidentiel"",""pptx""}",null,null,null
"""83f0b8c4-d6be-5789-8a8d-e4a472…",0,1.0,"{""presentation_veltis.pdf"",8,""Dates clés"",""Dates clés - 2011 : création de Veltis par Claire Dumont et Hugo Lefèvre. - 2014 : premier Veltis Famille livré. - 2017 : lancement du Veltis Pro L. - 2019 : ouverture du centre logistique de Marnay. - 2022 : lancement du Veltis Trike. - 2026 : présentation du plan stratégique « Cap 2028 ». À propos de Veltis - Fondée en 2011, Veltis SA conçoit et fabrique à Valmont des vélos cargo à assistance électrique pour les familles, les professionnels et les collectivités. L'entreprise emploie 182 personnes et distribue ses vélos dans 11 pays."",""pdf""}",null,null,null


## Reranking

La recherche sémantique fonctionne en encodant séparément, d'un côté nos chunks, de l'autre la question de l'utilisateur, puis en comparant les vecteurs obtenus. Le problème, c'est que chaque chunk est encodé une fois pour toutes, sans savoir quelle question lui sera posée : son embedding est nécessairement généraliste. Si nous connaissions à l'avance les questions des utilisateurs, nous encoderions les chunks autrement, en fonction de ces questions.

C'est un peu comme devoir expliquer un sujet sans connaître son public : faute de savoir à qui l'on s'adresse, on est forcé de produire une explication généraliste.

Le reranking corrige ce défaut en réordonnant les chunks par pertinence. Cette pertinence est calculée par un modèle appelé *cross-encoder* qui, contrairement à un modèle d'embedding, lit la question et chaque chunk ensemble et produit directement un score de pertinence pour la paire. Il peut ainsi juger si le chunk répond réellement à la question, et pas seulement s'il traite du même sujet.

Ce calcul se fait obligatoirement à la volée, puisqu'il dépend de la question : rien ne peut être préparé à l'avance. Il serait donc bien trop coûteux sur l'ensemble du corpus. C'est pourquoi on ne l'applique qu'aux quelques dizaines de chunks déjà présélectionnés par la recherche sémantique et lexicale, ce qui limite la latence ajoutée à un niveau généralement raisonnable. On ne conserve ensuite que les meilleurs chunks pour les transmettre au LLM.

Le SDK d'OpenAI n'a rien de prévu pour le reranking, il faut donc passer par des appels classiques d'API via l'endpoit ``/rerank``. Il suffit d'y envoyer le nom du modèle, notre requête et les documents

In [ ]:
EMBEDDING_URL = 'http://localhost:8000/v1'
EMBEDDING_MODEL = 'bge-embedder'

embedder = OpenAI(
    base_url=EMBEDDING_URL,
    api_key='no-sk'
)

RERANKING_URL = 'http://localhost:8001/rerank'
RERANKING_MODEL=  'bge-reranker'

In [54]:
query = 'Raconte moi la fin de Roméo et Juliette.'
documents = [
    # Pertinents : décrivent directement la fin
    "Et c'est ainsi que côte à côte ils moururent",
    "Croyant Juliette morte, Roméo boit le poison dans le tombeau des Capulet et meurt à ses côtés.",
    "À son réveil, Juliette découvre Roméo mort à ses côtés et se poignarde avec la dague de Roméo.",
    "Devant les corps des deux amants, les Montaigu et les Capulet se réconcilient enfin.",

    # Partiellement pertinent : prépare la fin sans la raconter
    "Frère Laurent donne à Juliette un breuvage qui la plongera dans un sommeil semblable à la mort.",

    # Pièges : même œuvre, mais pas la fin
    "Roméo et Juliette se rencontrent pour la première fois lors d'un bal chez les Capulet.",
    "Dans la scène du balcon, Juliette se demande pourquoi Roméo doit porter le nom de Montaigu.",
    "Mercutio est tué en duel par Tybalt, ce qui pousse Roméo à se venger.",

    # Pièges : même thème ou mêmes mots, mais autre sujet
    "C'est au dernier moment, pris d'un doute, qu'Orphée se retourna et perdit à jamais son amour.",
    "À la fin de Tristan et Iseult, les deux amants meurent l'un après l'autre, unis dans la mort.",
    "La pièce se termine sur la mort d'Hamlet et de presque toute la cour du Danemark.",
    "À Vérone, des milliers de touristes visitent chaque année la maison dite de Juliette.",

    # Hors sujet
    "Préchauffez le four à 180 °C et faites cuire le gâteau pendant 35 minutes.",
    "La fin de la saison de football est prévue pour la mi-mai.",
]

payload = {'model': RERANKING_MODEL, 'query': query, 'documents': documents}

r = rq.post(RERANKING_URL, json=payload)
data = pl.from_dicts(r.json()).unnest('results').unnest('document').sort('relevance_score', descending=True)
data

id,model,usage,index,text,multi_modal,relevance_score
str,str,struct[2],i64,str,null,f64
"""score-88600fa5780e0642""","""bge-reranker""","{570,570}",1,"""Croyant Juliette morte, Roméo …",null,0.436754
"""score-88600fa5780e0642""","""bge-reranker""","{570,570}",2,"""À son réveil, Juliette découvr…",null,0.412391
"""score-88600fa5780e0642""","""bge-reranker""","{570,570}",0,"""Et c'est ainsi que côte à côte…",null,0.042195
"""score-88600fa5780e0642""","""bge-reranker""","{570,570}",5,"""Roméo et Juliette se rencontre…",null,0.035028
"""score-88600fa5780e0642""","""bge-reranker""","{570,570}",4,"""Frère Laurent donne à Juliette…",null,0.004069
…,…,…,…,…,…,…
"""score-88600fa5780e0642""","""bge-reranker""","{570,570}",10,"""La pièce se termine sur la mor…",null,0.001052
"""score-88600fa5780e0642""","""bge-reranker""","{570,570}",8,"""C'est au dernier moment, pris …",null,0.000668
"""score-88600fa5780e0642""","""bge-reranker""","{570,570}",11,"""À Vérone, des milliers de tour…",null,0.000029


In [55]:
query_vector = np.array(embedder.embeddings.create(model=EMBEDDING_MODEL, input=query).data[0].embedding)
document_vectors = np.array([embedding.embedding for embedding in embedder.embeddings.create(model=EMBEDDING_MODEL, input=documents).data])

### Comparaison

On constate que certains documents n'ont pas du tout le même rang après le reranker.

In [56]:
cosine_scores = pl.DataFrame({'cosine_score': document_vectors @ query_vector}).with_row_index()
data.join(
    cosine_scores, on='index'
).with_columns(
    rank_reranker = pl.col('relevance_score').rank('ordinal', descending=True),
    rank_cosine = pl.col('cosine_score').rank('ordinal', descending=True)
).sort('relevance_score', descending=True)

id,model,usage,index,text,multi_modal,relevance_score,cosine_score,rank_reranker,rank_cosine
str,str,struct[2],i64,str,null,f64,f64,u32,u32
"""score-88600fa5780e0642""","""bge-reranker""","{570,570}",1,"""Croyant Juliette morte, Roméo …",null,0.436754,0.621596,1,3
"""score-88600fa5780e0642""","""bge-reranker""","{570,570}",2,"""À son réveil, Juliette découvr…",null,0.412391,0.642245,2,2
"""score-88600fa5780e0642""","""bge-reranker""","{570,570}",0,"""Et c'est ainsi que côte à côte…",null,0.042195,0.443334,3,13
"""score-88600fa5780e0642""","""bge-reranker""","{570,570}",5,"""Roméo et Juliette se rencontre…",null,0.035028,0.644563,4,1
"""score-88600fa5780e0642""","""bge-reranker""","{570,570}",4,"""Frère Laurent donne à Juliette…",null,0.004069,0.514886,5,9
…,…,…,…,…,…,…,…,…,…
"""score-88600fa5780e0642""","""bge-reranker""","{570,570}",10,"""La pièce se termine sur la mor…",null,0.001052,0.538328,10,6
"""score-88600fa5780e0642""","""bge-reranker""","{570,570}",8,"""C'est au dernier moment, pris …",null,0.000668,0.520915,11,8
"""score-88600fa5780e0642""","""bge-reranker""","{570,570}",11,"""À Vérone, des milliers de tour…",null,0.000029,0.473401,12,11
